<a href="https://colab.research.google.com/github/JaimeTS/hyrox-analysis/blob/main/notebooks/01_scraping_hyrox.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# Scraping de resultados HYROX

Objetivo: descargar los resultados de una prueba concreta desde HyResult y convertirlos en una base de datos estructurada para análisis de rendimiento, pacing y diferencias por edad y sexo.

## 1. Carga de librerías

Cargamos las librerías necesarias para descargar páginas web, leer HTML, estructurar datos en tablas y controlar pausas entre peticiones.

In [ ]:
import requests
from bs4 import BeautifulSoup
import pandas as pd
import numpy as np
import math
import time

## 2. Parámetros iniciales

Definimos la URL base de HyResult, la URL del evento y la URL de la categoría que vamos a descargar inicialmente.

In [ ]:
BASE_URL = "https://www.hyresult.com"

event_url = "https://www.hyresult.com/event/s8-2026-barcelona"
ranking_url = "https://www.hyresult.com/ranking/s8-2026-barcelona-hyrox-men"

headers = {
    "User-Agent": "Mozilla/5.0"
}

## 3. Extracción de enlaces a rankings del evento

La página del evento contiene enlaces a las distintas divisiones de la prueba. Extraemos esos enlaces para poder automatizar posteriormente la descarga de varias categorías.

In [ ]:
response_event = requests.get(event_url, headers=headers)
soup_event = BeautifulSoup(response_event.text, "html.parser")

links = []

for a in soup_event.find_all("a", href=True):
    href = a["href"]
    texto = a.get_text(strip=True)
    links.append((texto, href))

links_df = pd.DataFrame(links, columns=["texto", "href"])

ranking_links = links_df[
    links_df["href"].str.contains("/ranking/", case=False, na=False)
].drop_duplicates()

ranking_links

,texto,href
40,Ranking,/ranking/s8-2026-barcelona-hyrox-men
41,Ranking,/ranking/s8-2026-barcelona-hyrox-women
42,Ranking,/ranking/s8-2026-barcelona-hyrox-doubles-men
43,Ranking,/ranking/s8-2026-barcelona-hyrox-doubles-women
44,Ranking,/ranking/s8-2026-barcelona-hyrox-doubles-mixed
45,Ranking,/ranking/s8-2026-barcelona-hyrox-pro-men
46,Ranking,/ranking/s8-2026-barcelona-hyrox-pro-women
47,Ranking,/ranking/s8-2026-barcelona-hyrox-pro-doubles-men
48,Ranking,/ranking/s8-2026-barcelona-hyrox-pro-doubles-w...
49,Ranking,/ranking/s8-2026-barcelona-hyrox-team-relay-men


## 4. Función robusta para extraer una página del ranking

HyResult presenta normalmente un atleta por fila, aunque algunas posiciones aparecen divididas en varias filas consecutivas. Esta función reconstruye automáticamente esos casos para obtener todos los participantes de la página.

In [ ]:
def extraer_ranking_pagina(url):
    response = requests.get(url, headers=headers)
    soup = BeautifulSoup(response.text, "html.parser")

    rows = soup.find_all("tr")
    resultados = []

    i = 0

    while i < len(rows):
        cells = rows[i].find_all("td")
        textos = [c.get_text(" ", strip=True) for c in cells]

        # ==========================================================
        # CASO NORMAL
        # ==========================================================
        if len(cells) >= 7:

            posicion = cells[1].get_text(strip=True)
            posicion_ag = cells[2].get_text(strip=True)

            name_cell = cells[3]
            link_athlete = name_cell.find("a")
            img_flag = name_cell.find("img")

            nombre = link_athlete.get_text(strip=True) if link_athlete else None
            athlete_url = (
                link_athlete["href"]
                if link_athlete and link_athlete.has_attr("href")
                else None
            )

            pais = (
                img_flag["title"]
                if img_flag and img_flag.has_attr("title")
                else None
            )

            grupo_edad = cells[4].get_text(strip=True)
            tiempo = cells[5].get_text(strip=True)

            analyze_link = cells[6].find("a")
            analyze_url = (
                analyze_link["href"]
                if analyze_link and analyze_link.has_attr("href")
                else None
            )

            if nombre and tiempo and analyze_url:

                resultados.append({
                    "posicion": posicion,
                    "posicion_grupo_edad": posicion_ag,
                    "nombre": nombre,
                    "pais": pais,
                    "grupo_edad": grupo_edad,
                    "tiempo": tiempo,
                    "athlete_url": athlete_url,
                    "analyze_url": analyze_url
                })

            i += 1
            continue

        # ==========================================================
        # CASO ESPECIAL A
        # ==========================================================
        if len(cells) >= 3 and i + 4 < len(rows):

            posicion = cells[1].get_text(strip=True)
            posicion_ag = cells[2].get_text(strip=True)

            row_nombre = rows[i + 1]
            row_grupo = rows[i + 2]
            row_tiempo = rows[i + 3]
            row_analyze = rows[i + 4]

            nombre_cell = row_nombre.find("td")
            link_athlete = nombre_cell.find("a") if nombre_cell else None
            img_flag = nombre_cell.find("img") if nombre_cell else None

            nombre = link_athlete.get_text(strip=True) if link_athlete else None
            athlete_url = (
                link_athlete["href"]
                if link_athlete and link_athlete.has_attr("href")
                else None
            )

            pais = (
                img_flag["title"]
                if img_flag and img_flag.has_attr("title")
                else None
            )

            grupo_edad = row_grupo.get_text(" ", strip=True)
            tiempo = row_tiempo.get_text(" ", strip=True)

            analyze_link = row_analyze.find("a")
            analyze_url = (
                analyze_link["href"]
                if analyze_link and analyze_link.has_attr("href")
                else None
            )

            if nombre and tiempo and analyze_url:

                resultados.append({
                    "posicion": posicion,
                    "posicion_grupo_edad": posicion_ag,
                    "nombre": nombre,
                    "pais": pais,
                    "grupo_edad": grupo_edad,
                    "tiempo": tiempo,
                    "athlete_url": athlete_url,
                    "analyze_url": analyze_url
                })

                i += 5
                continue

        # ==========================================================
        # CASO ESPECIAL C
        # (fila vacía + datos repartidos)
        # ==========================================================
        if len(cells) == 1 and textos == [""] and i + 6 < len(rows):

            row_pos = rows[i + 1]
            row_pos_ag = rows[i + 2]
            row_nombre = rows[i + 3]
            row_grupo = rows[i + 4]
            row_tiempo = rows[i + 5]
            row_analyze = rows[i + 6]

            posicion = row_pos.get_text(" ", strip=True)
            posicion_ag = row_pos_ag.get_text(" ", strip=True)
            nombre = row_nombre.get_text(" ", strip=True)
            grupo_edad = row_grupo.get_text(" ", strip=True)
            tiempo = row_tiempo.get_text(" ", strip=True)

            analyze_link = row_analyze.find("a")
            analyze_url = (
                analyze_link["href"]
                if analyze_link and analyze_link.has_attr("href")
                else None
            )

            if (
                posicion.isdigit()
                and posicion_ag.isdigit()
                and nombre
                and tiempo
                and analyze_url
            ):

                resultados.append({
                    "posicion": posicion,
                    "posicion_grupo_edad": posicion_ag,
                    "nombre": nombre,
                    "pais": None,
                    "grupo_edad": grupo_edad,
                    "tiempo": tiempo,
                    "athlete_url": None,
                    "analyze_url": analyze_url
                })

                i += 7
                continue

        # ==========================================================
        # CASO ESPECIAL B
        # ==========================================================
        if len(cells) >= 2 and i + 5 < len(rows):

            posicion = cells[1].get_text(strip=True)

            row_pos_ag = rows[i + 1]
            row_nombre = rows[i + 2]
            row_grupo = rows[i + 3]
            row_tiempo = rows[i + 4]
            row_analyze = rows[i + 5]

            posicion_ag = row_pos_ag.get_text(" ", strip=True)

            nombre_cell = row_nombre.find("td")
            link_athlete = nombre_cell.find("a") if nombre_cell else None
            img_flag = nombre_cell.find("img") if nombre_cell else None

            nombre = link_athlete.get_text(strip=True) if link_athlete else None
            athlete_url = (
                link_athlete["href"]
                if link_athlete and link_athlete.has_attr("href")
                else None
            )

            pais = (
                img_flag["title"]
                if img_flag and img_flag.has_attr("title")
                else None
            )

            grupo_edad = row_grupo.get_text(" ", strip=True)
            tiempo = row_tiempo.get_text(" ", strip=True)

            analyze_link = row_analyze.find("a")
            analyze_url = (
                analyze_link["href"]
                if analyze_link and analyze_link.has_attr("href")
                else None
            )

            if nombre and tiempo and analyze_url:

                resultados.append({
                    "posicion": posicion,
                    "posicion_grupo_edad": posicion_ag,
                    "nombre": nombre,
                    "pais": pais,
                    "grupo_edad": grupo_edad,
                    "tiempo": tiempo,
                    "athlete_url": athlete_url,
                    "analyze_url": analyze_url
                })

                i += 6
                continue

        i += 1

    df = pd.DataFrame(resultados)

    if len(df) > 0:

        df["athlete_url"] = df["athlete_url"].apply(
            lambda x: BASE_URL + x if isinstance(x, str) and x.startswith("/") else x
        )

        df["analyze_url"] = df["analyze_url"].apply(
            lambda x: BASE_URL + x if isinstance(x, str) and x.startswith("/") else x
        )

    return df

## 5. Prueba de extracción de una página

Probamos la función con la primera página y comprobamos que recupera las 100 posiciones.

In [ ]:
df_pagina_1 = extraer_ranking_pagina(ranking_url)

print("Filas extraídas:", len(df_pagina_1))

df_pagina_1.head(10)

Filas extraídas: 100


,posicion,posicion_grupo_edad,nombre,pais,grupo_edad,tiempo,athlete_url,analyze_url
0,1,1,Gaetan Gianni,France,16-24,53:44,https://www.hyresult.com/athlete/gaetan-gianni,https://www.hyresult.com/result/LR3MS4JI502C48
1,2,1,Emilio Aguayo,Spain,35-39,56:38,https://www.hyresult.com/athlete/emilio-aguayo,https://www.hyresult.com/result/LR3MS4JI501875
2,3,1,Loic Cebelieu,France,25-29,57:17,https://www.hyresult.com/athlete/loic-cebelieu,https://www.hyresult.com/result/LR3MS4JI5010E6
3,4,1,Ferran Bochaca Sabarich,Spain,30-34,57:24,https://www.hyresult.com/athlete/ferran-bochac...,https://www.hyresult.com/result/LR3MS4JI501D84
4,5,2,Alberto Casado Aroca,Spain,30-34,57:38,https://www.hyresult.com/athlete/alberto-casad...,https://www.hyresult.com/result/LR3MS4JI502972
5,6,3,Sebastien Rajkowski,France,30-34,58:10,https://www.hyresult.com/athlete/sebastien-raj...,https://www.hyresult.com/result/LR3MS4JI5022B0
6,7,4,Mehdi Berrouigat,France,30-34,58:19,https://www.hyresult.com/athlete/mehdi-berrouigat,https://www.hyresult.com/result/LR3MS4JI503639
7,8,2,Pau Nacenta Merinas,Spain,25-29,58:35,https://www.hyresult.com/athlete/pau-nacenta-m...,https://www.hyresult.com/result/LR3MS4JI50188C
8,9,3,Aaron Timmerman,Belgium,25-29,58:38,https://www.hyresult.com/athlete/aaron-timmerman,https://www.hyresult.com/result/LR3MS4JI500DE4
9,10,4,Walter Paladina,Italy,25-29,58:41,https://www.hyresult.com/athlete/walter-paladina,https://www.hyresult.com/result/LR3MS4JI502477


## 6. Función para descargar una categoría completa

HyResult pagina los rankings en bloques de 100 participantes. Esta función recorre todas las páginas de una categoría y concatena los resultados en un único DataFrame.

In [ ]:
def descargar_categoria_completa(ranking_url, total_atletas, pausa=0.5):
    participantes_por_pagina = 100
    num_paginas = math.ceil(total_atletas / participantes_por_pagina)

    dfs = []

    for pagina in range(1, num_paginas + 1):
        url = f"{ranking_url}?p={pagina}"

        print(f"Descargando página {pagina}/{num_paginas}")

        df_pagina = extraer_ranking_pagina(url)
        dfs.append(df_pagina)

        time.sleep(pausa)

    df_categoria = pd.concat(dfs, ignore_index=True)

    return df_categoria

## 7. Descarga de HYROX Men Barcelona 2026

Aplicamos la función a la categoría HYROX Men del evento Barcelona 2026. En la página del evento esta categoría aparece con 2350 atletas.

In [ ]:
total_atletas = 2350

df_categoria = descargar_categoria_completa(
    ranking_url=ranking_url,
    total_atletas=total_atletas,
    pausa=0.5
)

print("Participantes extraídos:", len(df_categoria))

df_categoria.tail()

Descargando página 1/24
Descargando página 2/24
Descargando página 3/24
Descargando página 4/24
Descargando página 5/24
Descargando página 6/24
Descargando página 7/24
Descargando página 8/24
Descargando página 9/24
Descargando página 10/24
Descargando página 11/24
Descargando página 12/24
Descargando página 13/24
Descargando página 14/24
Descargando página 15/24
Descargando página 16/24
Descargando página 17/24
Descargando página 18/24
Descargando página 19/24
Descargando página 20/24
Descargando página 21/24
Descargando página 22/24
Descargando página 23/24
Descargando página 24/24
Participantes extraídos: 2349


,posicion,posicion_grupo_edad,nombre,pais,grupo_edad,tiempo,athlete_url,analyze_url
2344,2346,23,Carlos Barreto,Puerto Rico,60-64,2:36:53,https://www.hyresult.com/athlete/carlos-barreto,https://www.hyresult.com/result/LR3MS4JI5036F1
2345,2347,24,Emmanuel Weill,France,60-64,2:38:29,https://www.hyresult.com/athlete/emmanuel-weill,https://www.hyresult.com/result/LR3MS4JI503BBC
2346,2348,286,Mo Mo,United States,40-44,2:51:02,https://www.hyresult.com/athlete/mo-mo,https://www.hyresult.com/result/LR3MS4JI503D47
2347,2349,159,Nicolas Taillard,France,45-49,2:53:32,https://www.hyresult.com/athlete/nicolas-taillard,https://www.hyresult.com/result/LR3MS4JI501C3E
2348,2350,461,Avisek Mohanty,India,35-39,3:08:24,https://www.hyresult.com/athlete/avisek-mohanty,https://www.hyresult.com/result/LR3MS4JI50381A


## 8. Control de calidad de la extracción

Comprobamos si el número de participantes extraídos coincide con el total esperado y si hay posiciones duplicadas o faltantes.

In [ ]:
df_categoria["posicion"] = pd.to_numeric(df_categoria["posicion"], errors="coerce")
df_categoria["posicion_grupo_edad"] = pd.to_numeric(df_categoria["posicion_grupo_edad"], errors="coerce")

print("Total esperado:", total_atletas)
print("Total extraído:", len(df_categoria))

print("Posición mínima:", df_categoria["posicion"].min())
print("Posición máxima:", df_categoria["posicion"].max())

print("Posiciones duplicadas:", df_categoria["posicion"].duplicated().sum())

posiciones_esperadas = set(range(1, total_atletas + 1))
posiciones_observadas = set(df_categoria["posicion"].dropna().astype(int))

posiciones_faltantes = sorted(posiciones_esperadas - posiciones_observadas)

print("Número de posiciones faltantes:", len(posiciones_faltantes))
print("Primeras posiciones faltantes:", posiciones_faltantes[:20])

Total esperado: 2350
Total extraído: 2349
Posición mínima: 1
Posición máxima: 2350
Posiciones duplicadas: 0
Número de posiciones faltantes: 1
Primeras posiciones faltantes: [403]


EN HYROX BARCELONA 2026, EXISTE UN PROBLEMA CON LA POSICIÓN 403 DONDE NO APARECE CON LA DESCARGA NI EL NOMBRE NI EL ENLACE DEL ATLETA

## 9. Exportación de resultados

Guardamos la base de datos de la categoría en formato CSV para reutilizarla en análisis posteriores.

In [ ]:
nombre_archivo = "barcelona2026_hyrox_men.csv"

df_categoria.to_csv(
    nombre_archivo,
    index=False
)

print("Archivo guardado:", nombre_archivo)

Archivo guardado: barcelona2026_hyrox_men.csv


## 10. Descarga de la pestaña splits (datos individuales)

La página individual del atleta tiene varias pestañas. Los parciales parecen estar en la pestaña `Splits`, accesible mediante el parámetro `?tab=splits`.

In [90]:
## URL de splits de un atleta

url_resultado = df_categoria.loc[0, "analyze_url"]
url_splits = url_resultado + "?tab=splits"

response_splits = requests.get(url_splits, headers=headers)
soup_splits = BeautifulSoup(response_splits.text, "html.parser")

texto_splits = soup_splits.get_text(" | ", strip=True)

print("URL:", url_splits)
print("Código:", response_splits.status_code)
print("Longitud HTML:", len(response_splits.text))
print(texto_splits[:4000])

URL: https://www.hyresult.com/result/LR3MS4JI502C48?tab=splits
Código: 200
Longitud HTML: 193179
Home | Events | Compare | Rankings | All Time Ranking | Legends Ranking | Elite Points Ranking | Simulator | Locations | Toggle navigation | Search | Loading data ... | WE 💛 HYROX! | HYRESULT is your source for all HYROX data. Find HYROX rankings, athlete profiles, race analytics, start lists by wave, detailed comparisons and race time simulators. | App | Home | Compare | Events | Simulator | Locations | Rankings | Station guides | HYROX Guide | Run | SkiErg | Sled Push | Sled Pull | Burpee Broad Jump | Row | Farmers Carry | Sandbag Lunges | Wall Balls | Company & legal | About | Partner | Privacy | Legal | Connect | Contact | @hyresult | © | 2026 | HYRESULT GmbH. All rights reserved. | Home | Events | Compare | Rankings | All Time Ranking | Legends Ranking | Elite Points Ranking | Simulator | Locations | HYROX race analysis for Gaetan Gianni at HYROX Barcelona 2026 • HYRESULT | Events | / 

In [91]:
palabras_clave_splits = [
    "Run 1",
    "SkiErg",
    "Run 2",
    "Sled Push",
    "Run 3",
    "Sled Pull",
    "Run 4",
    "Burpee Broad Jump",
    "Run 5",
    "Row",
    "Run 6",
    "Farmers Carry",
    "Run 7",
    "Sandbag Lunges",
    "Run 8",
    "Wall Balls",
    "Roxzone"
]

for palabra in palabras_clave_splits:
    indice = texto_splits.find(palabra)
    print("\n" + "="*80)
    print("Palabra:", palabra)
    print("Índice:", indice)

    if indice != -1:
        print(texto_splits[indice:indice+1000])


Palabra: Run 1
Índice: -1

Palabra: SkiErg
Índice: 456
SkiErg | Sled Push | Sled Pull | Burpee Broad Jump | Row | Farmers Carry | Sandbag Lunges | Wall Balls | Company & legal | About | Partner | Privacy | Legal | Connect | Contact | @hyresult | © | 2026 | HYRESULT GmbH. All rights reserved. | Home | Events | Compare | Rankings | All Time Ranking | Legends Ranking | Elite Points Ranking | Simulator | Locations | HYROX race analysis for Gaetan Gianni at HYROX Barcelona 2026 • HYRESULT | Events | / | Barcelona 2026 | / | MEN | Gaetan Gianni | 53:44 | # | 1 | of | 2350 | # | 1 | in AG | 16-24 | Loading data ... | Totals | Runs | Workouts | Splits | Simulator | Split | Time of Day | Time | Diff | Roxzone In | 08:53:23 | 03:21 | 03:21 | SkiErg In | 08:53:26 | 03:23 | 0:02 | SkiErg Out | 08:57:25 | 07:23 | 04:00 | Roxzone Out | 08:57:28 | 07:25 | 0:02 | Roxzone In | 09:00:50 | 10:47 | 03:22 | Sled Push In | 09:01:02 | 11:00 | 0:13 | Sled Push Out | 09:03:10 | 13:07 | 02:07 | Roxzone Out | 0

## 11. Extracción de Splits de un atleta

Cada página de resultado contiene una pestaña Splits con el detalle completo de los tiempos acumulados y parciales de carrera y estaciones.

En este bloque extraemos la tabla de splits de un atleta individual.

In [92]:
url_resultado = df_categoria.loc[0, "analyze_url"]

url_splits = url_resultado + "?tab=splits"

response_splits = requests.get(url_splits, headers=headers)

print("Código:", response_splits.status_code)
print("Longitud HTML:", len(response_splits.text))

Código: 200
Longitud HTML: 193179


In [94]:
from io import StringIO

# Convertimos el HTML en tablas de pandas
tablas = pd.read_html(StringIO(response_splits.text))

print("Número de tablas encontradas:", len(tablas))

for i, tabla in enumerate(tablas):

    print("\n" + "="*80)
    print(f"TABLA {i}")
    print("Dimensiones:", tabla.shape)

    display(tabla.head(20))

Número de tablas encontradas: 20

TABLA 0
Dimensiones: (11, 4)


,Split,Time of Day,Time,Diff
0,Roxzone In,08:53:23,03:21,03:21
1,SkiErg In,08:53:26,03:23,0:02
2,SkiErg Out,08:57:25,07:23,04:00
3,Roxzone Out,08:57:28,07:25,0:02
4,Roxzone In,09:00:50,10:47,03:22
5,Sled Push In,09:01:02,11:00,0:13
6,Sled Push Out,09:03:10,13:07,02:07
7,Roxzone Out,09:03:13,13:11,0:04
8,Roxzone In,09:06:43,16:41,03:30
9,Sled Pull In,09:06:58,16:56,0:15



TABLA 1
Dimensiones: (1, 4)


,0,1,2,3
0,Roxzone Out,09:10:02,19:59,0:11



TABLA 2
Dimensiones: (1, 4)


...

,0,1,2,3
0,Roxzone Out,09:36:40,46:38,0:36



TABLA 18
Dimensiones: (1, 4)


,0,1,2,3
0,Wall Balls In,09:40:07,50:05,03:27



TABLA 19
Dimensiones: (1, 4)


,0,1,2,3
0,Total time,09:43:46,53:44,03:39


## 12. Reconstrucción de la tabla completa de splits

HYRESULT divide los splits en varias tablas HTML. En este bloque las unimos en una única tabla.

In [95]:
from io import StringIO

def extraer_splits(url_resultado):

    url_splits = url_resultado + "?tab=splits"

    response = requests.get(url_splits, headers=headers)

    tablas = pd.read_html(StringIO(response.text))

    tablas_limpias = []

    for i, tabla in enumerate(tablas):

        if i == 0:
            tablas_limpias.append(tabla)

        else:

            tabla.columns = ["Split", "Time of Day", "Time", "Diff"]

            tablas_limpias.append(tabla)

    df_splits = pd.concat(tablas_limpias, ignore_index=True)

    return df_splits

In [96]:
url_resultado = df_categoria.loc[0, "analyze_url"]

df_splits = extraer_splits(url_resultado)

print("Número de filas:", len(df_splits))

display(df_splits)

Número de filas: 30


,Split,Time of Day,Time,Diff
0,Roxzone In,08:53:23,03:21,03:21
1,SkiErg In,08:53:26,03:23,0:02
2,SkiErg Out,08:57:25,07:23,04:00
3,Roxzone Out,08:57:28,07:25,0:02
4,Roxzone In,09:00:50,10:47,03:22
5,Sled Push In,09:01:02,11:00,0:13
6,Sled Push Out,09:03:10,13:07,02:07
7,Roxzone Out,09:03:13,13:11,0:04
8,Roxzone In,09:06:43,16:41,03:30
9,Sled Pull In,09:06:58,16:56,0:15


## 13. Función para extraer splits con identificador del atleta

Convertimos la extracción anterior en una función reutilizable. La función descarga la pestaña Splits de un atleta y añade sus datos identificativos.

In [97]:
def extraer_splits_atleta(fila_atleta):
    url_resultado = fila_atleta["analyze_url"]
    url_splits = url_resultado + "?tab=splits"

    response = requests.get(url_splits, headers=headers)

    tablas = pd.read_html(StringIO(response.text))

    tablas_limpias = []

    for i, tabla in enumerate(tablas):
        if i == 0:
            tabla.columns = ["Split", "Time of Day", "Time", "Diff"]
            tablas_limpias.append(tabla)
        else:
            tabla.columns = ["Split", "Time of Day", "Time", "Diff"]
            tablas_limpias.append(tabla)

    df_splits = pd.concat(tablas_limpias, ignore_index=True)

    df_splits["posicion"] = fila_atleta["posicion"]
    df_splits["nombre"] = fila_atleta["nombre"]
    df_splits["grupo_edad"] = fila_atleta["grupo_edad"]
    df_splits["pais"] = fila_atleta["pais"]
    df_splits["tiempo_total"] = fila_atleta["tiempo"]
    df_splits["analyze_url"] = fila_atleta["analyze_url"]

    return df_splits

In [98]:
df_splits_1 = extraer_splits_atleta(df_categoria.iloc[0])

print(df_splits_1.shape)

df_splits_1.head(10)

(30, 10)


,Split,Time of Day,Time,Diff,posicion,nombre,grupo_edad,pais,tiempo_total,analyze_url
0,Roxzone In,08:53:23,03:21,03:21,1,Gaetan Gianni,16-24,France,53:44,https://www.hyresult.com/result/LR3MS4JI502C48
1,SkiErg In,08:53:26,03:23,0:02,1,Gaetan Gianni,16-24,France,53:44,https://www.hyresult.com/result/LR3MS4JI502C48
2,SkiErg Out,08:57:25,07:23,04:00,1,Gaetan Gianni,16-24,France,53:44,https://www.hyresult.com/result/LR3MS4JI502C48
3,Roxzone Out,08:57:28,07:25,0:02,1,Gaetan Gianni,16-24,France,53:44,https://www.hyresult.com/result/LR3MS4JI502C48
4,Roxzone In,09:00:50,10:47,03:22,1,Gaetan Gianni,16-24,France,53:44,https://www.hyresult.com/result/LR3MS4JI502C48
5,Sled Push In,09:01:02,11:00,0:13,1,Gaetan Gianni,16-24,France,53:44,https://www.hyresult.com/result/LR3MS4JI502C48
6,Sled Push Out,09:03:10,13:07,02:07,1,Gaetan Gianni,16-24,France,53:44,https://www.hyresult.com/result/LR3MS4JI502C48
7,Roxzone Out,09:03:13,13:11,0:04,1,Gaetan Gianni,16-24,France,53:44,https://www.hyresult.com/result/LR3MS4JI502C48
8,Roxzone In,09:06:43,16:41,03:30,1,Gaetan Gianni,16-24,France,53:44,https://www.hyresult.com/result/LR3MS4JI502C48
9,Sled Pull In,09:06:58,16:56,0:15,1,Gaetan Gianni,16-24,France,53:44,https://www.hyresult.com/result/LR3MS4JI502C48


## 14. Prueba de extracción de splits para varios atletas

Antes de descargar todos los atletas, probamos la función con una muestra pequeña para comprobar que la estructura es estable.

In [99]:
# Probamos con los 5 primeros atletas

dfs_splits_prueba = []

for i in range(5):
    print("Extrayendo atleta", i + 1)

    df_temp = extraer_splits_atleta(df_categoria.iloc[i])
    dfs_splits_prueba.append(df_temp)

    time.sleep(0.5)

df_splits_prueba = pd.concat(dfs_splits_prueba, ignore_index=True)

print("Filas totales:", len(df_splits_prueba))

df_splits_prueba.head()

Extrayendo atleta 1
Extrayendo atleta 2
Extrayendo atleta 3
Extrayendo atleta 4
Extrayendo atleta 5
Filas totales: 150


,Split,Time of Day,Time,Diff,posicion,nombre,grupo_edad,pais,tiempo_total,analyze_url
0,Roxzone In,08:53:23,03:21,03:21,1,Gaetan Gianni,16-24,France,53:44,https://www.hyresult.com/result/LR3MS4JI502C48
1,SkiErg In,08:53:26,03:23,0:02,1,Gaetan Gianni,16-24,France,53:44,https://www.hyresult.com/result/LR3MS4JI502C48
2,SkiErg Out,08:57:25,07:23,04:00,1,Gaetan Gianni,16-24,France,53:44,https://www.hyresult.com/result/LR3MS4JI502C48
3,Roxzone Out,08:57:28,07:25,0:02,1,Gaetan Gianni,16-24,France,53:44,https://www.hyresult.com/result/LR3MS4JI502C48
4,Roxzone In,09:00:50,10:47,03:22,1,Gaetan Gianni,16-24,France,53:44,https://www.hyresult.com/result/LR3MS4JI502C48


## 15. Prueba ampliada con 50 atletas

Antes de descargar toda la categoría, probamos con 50 atletas para comprobar estabilidad y detectar posibles páginas sin splits o estructuras diferentes.

In [101]:
dfs_splits_50 = []
errores_splits = []

for i in range(50):
    fila = df_categoria.iloc[i]

    try:
        print(f"Extrayendo {i+1}/50: {fila['nombre']}")

        df_temp = extraer_splits_atleta(fila)
        dfs_splits_50.append(df_temp)

        time.sleep(0.5)

    except Exception as e:
        print(f"Error en {fila['nombre']}: {e}")

        errores_splits.append({
            "indice": i,
            "posicion": fila["posicion"],
            "nombre": fila["nombre"],
            "analyze_url": fila["analyze_url"],
            "error": str(e)
        })

df_splits_50 = pd.concat(dfs_splits_50, ignore_index=True)

print("Atletas con splits descargados:", df_splits_50["nombre"].nunique())
print("Filas totales:", len(df_splits_50))
print("Errores:", len(errores_splits))

df_splits_50.head()

Extrayendo 1/50: Gaetan Gianni
Extrayendo 2/50: Emilio Aguayo
Extrayendo 3/50: Loic Cebelieu
Extrayendo 4/50: Ferran Bochaca Sabarich
Extrayendo 5/50: Alberto Casado Aroca
Extrayendo 6/50: Sebastien Rajkowski
Extrayendo 7/50: Mehdi Berrouigat
Extrayendo 8/50: Pau Nacenta Merinas
Extrayendo 9/50: Aaron Timmerman
Extrayendo 10/50: Walter Paladina
Extrayendo 11/50: Ethan Smith
Extrayendo 12/50: Mohamed Yassin Kattar
Extrayendo 13/50: Yoel Ferrer
Extrayendo 14/50: Russell Christie
Extrayendo 15/50: Martin Casse
Extrayendo 16/50: James Francis
Extrayendo 17/50: Thomas Jurado
Extrayendo 18/50: Robert Stols
Extrayendo 19/50: Sam Booth
Extrayendo 20/50: David San Miguel López
Extrayendo 21/50: Xabier De Velasco
Extrayendo 22/50: Javier Puyalto Monaj
Extrayendo 23/50: Ivan Llorens Catala
Extrayendo 24/50: Sam Garcia
Extrayendo 25/50: John Murtagh
Extrayendo 26/50: Julien Caujolle
Extrayendo 27/50: Luis Caja Nacher
Extrayendo 28/50: Joost Huisink
Extrayendo 29/50: Yann Borderie
Extrayendo 30/50:

,Split,Time of Day,Time,Diff,posicion,nombre,grupo_edad,pais,tiempo_total,analyze_url
0,Roxzone In,08:53:23,03:21,03:21,1,Gaetan Gianni,16-24,France,53:44,https://www.hyresult.com/result/LR3MS4JI502C48
1,SkiErg In,08:53:26,03:23,0:02,1,Gaetan Gianni,16-24,France,53:44,https://www.hyresult.com/result/LR3MS4JI502C48
2,SkiErg Out,08:57:25,07:23,04:00,1,Gaetan Gianni,16-24,France,53:44,https://www.hyresult.com/result/LR3MS4JI502C48
3,Roxzone Out,08:57:28,07:25,0:02,1,Gaetan Gianni,16-24,France,53:44,https://www.hyresult.com/result/LR3MS4JI502C48
4,Roxzone In,09:00:50,10:47,03:22,1,Gaetan Gianni,16-24,France,53:44,https://www.hyresult.com/result/LR3MS4JI502C48


In [102]:
df_splits_50.groupby("nombre")["Split"].count().sort_values()

,Split
nombre,
Aaron Timmerman,30
Alberto Casado Aroca,30
Alessandro Rossi,30
Alex Lang,30
Ansumana Konteh Sanneh,30
Axel Solegre,30
Barney Fairnington,30
Colin Davy,30
David San Miguel López,30


In [104]:
df_categoria.to_csv("barcelona2026_hyrox_men_ranking.csv", index=False)

In [105]:
#DESCARGA COMPLETA
dfs_splits_todos = []
errores_splits_todos = []

for i in range(len(df_categoria)):
    fila = df_categoria.iloc[i]

    try:
        print(f"Extrayendo {i+1}/{len(df_categoria)}: {fila['nombre']}")

        df_temp = extraer_splits_atleta(fila)
        dfs_splits_todos.append(df_temp)

        time.sleep(0.5)

    except Exception as e:
        print(f"Error en {fila['nombre']}: {e}")

        errores_splits_todos.append({
            "indice": i,
            "posicion": fila["posicion"],
            "nombre": fila["nombre"],
            "analyze_url": fila["analyze_url"],
            "error": str(e)
        })

df_splits_todos = pd.concat(dfs_splits_todos, ignore_index=True)

print("Atletas con splits descargados:", df_splits_todos["nombre"].nunique())
print("Filas totales:", len(df_splits_todos))
print("Errores:", len(errores_splits_todos))

Extrayendo 1/2349: Gaetan Gianni
Extrayendo 2/2349: Emilio Aguayo
Extrayendo 3/2349: Loic Cebelieu
Extrayendo 4/2349: Ferran Bochaca Sabarich
Extrayendo 5/2349: Alberto Casado Aroca
Extrayendo 6/2349: Sebastien Rajkowski
Extrayendo 7/2349: Mehdi Berrouigat
Extrayendo 8/2349: Pau Nacenta Merinas
Extrayendo 9/2349: Aaron Timmerman
Extrayendo 10/2349: Walter Paladina
Extrayendo 11/2349: Ethan Smith
Extrayendo 12/2349: Mohamed Yassin Kattar
Extrayendo 13/2349: Yoel Ferrer
Extrayendo 14/2349: Russell Christie
Extrayendo 15/2349: Martin Casse
Extrayendo 16/2349: James Francis
Extrayendo 17/2349: Thomas Jurado
Extrayendo 18/2349: Robert Stols
Extrayendo 19/2349: Sam Booth
Extrayendo 20/2349: David San Miguel López
Extrayendo 21/2349: Xabier De Velasco
Extrayendo 22/2349: Javier Puyalto Monaj
Extrayendo 23/2349: Ivan Llorens Catala
Extrayendo 24/2349: Sam Garcia
Extrayendo 25/2349: John Murtagh
Extrayendo 26/2349: Julien Caujolle
Extrayendo 27/2349: Luis Caja Nacher
Extrayendo 28/2349: Joost H

In [106]:
df_splits_todos.to_csv("barcelona2026_hyrox_men_splits.csv", index=False)